# Silver: conformed and quality-checked

Turns two systems that disagree into one vocabulary, and records every row it had to
reject. Bronze is what arrived; Silver is what we are willing to stand behind.

| | |
| --- | --- |
| **Reads** | seven `bronze_*` tables |
| **Writes** | conformed dimensions and facts, plus `silver_data_quality` and `silver_quarantine` |

### What Silver actually has to fix

| Problem in the extract | Handling |
| --- | --- |
| Two date formats (`dd/mm/yyyy` and `mm/dd/yyyy`) | Parsed per source system, never guessed |
| Money as `"$1,234.56"` text | Stripped and cast to decimal |
| `RGI` / `rgi` / `Rent-Geared-to-Income` | Mapped to one controlled value |
| Ward name casing inconsistent | Title-cased once |
| A re-sent receipts batch | Deduplicated on the natural key |
| Receipts for unknown accounts | **Quarantined, not dropped** |
| Work orders with unresolvable units | **Quarantined, not dropped** |

Quarantine matters more than it looks. Silently dropping an orphan receipt makes
arrears look worse than it is, and nobody can tell afterwards that it happened.

In [ ]:
PIPELINE_RUN_ID = ""

In [ ]:
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DecimalType, DoubleType, IntegerType, StringType, StructField, StructType,
)

# --- cross-lakehouse reads -------------------------------------------------------
# spark.read.table() resolves only against this notebook's default lakehouse.
_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table_name):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table_name}")


def bronze(name):
    return lake_table("bronze_lakehouse", name)


RUN_ID = PIPELINE_RUN_ID or (bronze("bronze_pms_units")
                             .orderBy(F.desc("extracted_at"))
                             .select("run_id").first()["run_id"])
GENERATED_AT = datetime.now(timezone.utc).isoformat()
print("run_id:", RUN_ID)

quality_rows, quarantine_rows = [], []


def check(table_name, rule, passed, failed, note=None):
    quality_rows.append({
        "run_id": RUN_ID, "table_name": table_name, "rule": rule,
        "rows_passed": int(passed), "rows_failed": int(failed),
        "pass_rate": round(passed / (passed + failed), 4) if (passed + failed) else 1.0,
        "note": note, "checked_at_utc": GENERATED_AT,
    })


def quarantine(frame, source_table, reason):
    rows = frame.limit(500).collect()
    for row in rows:
        quarantine_rows.append({
            "run_id": RUN_ID, "source_table": source_table, "reason": reason,
            "record": json.dumps({k: (None if v is None else str(v))
                                  for k, v in row.asDict().items()})[:1500],
            "quarantined_at_utc": GENERATED_AT,
        })



def title_case_hyphenated(column):
    """Title-case a name whose parts may be separated by hyphens as well as spaces."""
    spaced = F.regexp_replace(F.col(column), "-", " - ")
    return F.regexp_replace(F.initcap(F.trim(spaced)), " - ", "-")


def money(column):
    """'$1,234.56' -> 1234.56 . Currency symbols and separators only, no rounding."""
    return F.regexp_replace(F.col(column), r"[$,\s]", "").cast(DecimalType(12, 2))

## 1. Dimensions

In [ ]:
buildings = (bronze("bronze_pms_buildings").filter(F.col("run_id") == RUN_ID)
             .select(
                 F.col("building_id"),
                 F.initcap(F.trim(F.col("building_name"))).alias("building_name"),
                 # initcap only capitalises after whitespace, so a hyphenated
                 # ward such as EGLINTON-LAWRENCE would come out as
                 # "Eglinton-lawrence". Capitalise each hyphenated part.
                 title_case_hyphenated("ward_name").alias("ward_name"),
                 F.initcap(F.trim(F.col("region"))).alias("region"),
                 F.col("property_type"),
                 F.col("year_built").cast(IntegerType()).alias("year_built"),
                 F.col("total_units").cast(IntegerType()).alias("total_units"),
                 F.lit(RUN_ID).alias("run_id"),
                 F.lit(GENERATED_AT).alias("generated_at_utc")))

# One controlled vocabulary for tenure, from five spellings in the source.
tenure = (F.when(F.lower(F.trim(F.col("unit_type"))).isin("rgi", "rent-geared-to-income"),
                 F.lit("RGI"))
          .when(F.lower(F.trim(F.col("unit_type"))) == "market", F.lit("Market"))
          .otherwise(F.lit("Unknown")))

units = (bronze("bronze_pms_units").filter(F.col("run_id") == RUN_ID)
         .select(
             F.col("unit_id"), F.col("building_id"), F.col("unit_number"),
             F.col("bedroom_count").cast(IntegerType()).alias("bedroom_count"),
             tenure.alias("tenure_type"),
             F.when(F.col("accessible_flag") == "Y", F.lit(True))
              .when(F.col("accessible_flag") == "N", F.lit(False))
              .otherwise(F.lit(None)).alias("is_accessible"),
             F.lit(RUN_ID).alias("run_id"),
             F.lit(GENERATED_AT).alias("generated_at_utc")))

unknown_tenure = units.filter(F.col("tenure_type") == "Unknown").count()
check("silver_dim_unit", "tenure_type resolves to RGI or Market",
      units.count() - unknown_tenure, unknown_tenure,
      "source carries five spellings of two concepts")

tenancies = (bronze("bronze_pms_tenancies").filter(F.col("run_id") == RUN_ID)
             .select(
                 F.col("household_ref"), F.col("unit_id"),
                 F.col("household_size").cast(IntegerType()).alias("household_size"),
                 F.col("income_band"), F.col("subsidy_type"),
                 # PMS writes dd/MM/yyyy.
                 F.to_date(F.col("move_in_dt"), "dd/MM/yyyy").alias("move_in_date"),
                 F.to_date(F.col("move_out_dt"), "dd/MM/yyyy").alias("move_out_date"),
                 F.lit(RUN_ID).alias("run_id"),
                 F.lit(GENERATED_AT).alias("generated_at_utc")))

bad_dates = tenancies.filter(F.col("move_in_date").isNull()).count()
check("silver_dim_household", "move_in_date parses as dd/MM/yyyy",
      tenancies.count() - bad_dates, bad_dates)

## 2. Facts, and the two systems that disagree

Charges are keyed by `household_ref`, receipts by `tenant_account`. Same concept, two
names — conforming them is the whole job of this layer.

In [ ]:
charges = (bronze("bronze_pms_rent_charges").filter(F.col("run_id") == RUN_ID)
           .select(
               F.col("charge_id"), F.col("household_ref").alias("household_key"),
               F.col("unit_id"),
               F.to_date(F.concat_ws("-", F.col("charge_period"), F.lit("01")),
                         "yyyy-MM-dd").alias("period_start"),
               F.col("charge_type"),
               money("charge_amount").alias("charge_amount"),
               F.lit(RUN_ID).alias("run_id"),
               F.lit(GENERATED_AT).alias("generated_at_utc")))

bad_amounts = charges.filter(F.col("charge_amount").isNull()).count()
check("silver_fact_rent_charge", "charge_amount casts from currency text",
      charges.count() - bad_amounts, bad_amounts, "source format: \"$1,234.56\"")

receipts_raw = bronze("bronze_fin_receipts").filter(F.col("run_id") == RUN_ID)

# The re-sent batch. receipt_id is the natural key, so a repeat is a duplicate.
before = receipts_raw.count()
receipts_deduped = receipts_raw.dropDuplicates(["receipt_id"])
duplicates = before - receipts_deduped.count()
check("silver_fact_receipt", "receipt_id is unique",
      receipts_deduped.count(), duplicates,
      f"{duplicates} duplicate rows removed - re-sent extract batch")

receipts = (receipts_deduped.select(
    F.col("receipt_id"), F.col("tenant_account").alias("household_key"),
    # Finance writes MM/dd/yyyy. Parsing this with the PMS pattern would silently
    # transpose day and month for the first twelve days of every month.
    F.to_date(F.col("posting_date"), "MM/dd/yyyy").alias("posting_date"),
    money("amount_cad").alias("payment_amount"),
    F.col("payment_method"),
    F.lit(RUN_ID).alias("run_id"),
    F.lit(GENERATED_AT).alias("generated_at_utc")))

# Orphan receipts: quarantined, never dropped. Dropping them overstates arrears.
known_households = tenancies.select("household_ref").distinct()
orphan_receipts = receipts.join(known_households,
                                receipts.household_key == known_households.household_ref,
                                "left_anti")
orphan_count = orphan_receipts.count()
if orphan_count:
    quarantine(orphan_receipts, "silver_fact_receipt",
               "tenant_account not present in the tenancy extract")
check("silver_fact_receipt", "household_key resolves to a known tenancy",
      receipts.count() - orphan_count, orphan_count,
      "quarantined, not dropped - dropping would overstate arrears")
receipts_valid = receipts.join(known_households,
                               receipts.household_key == known_households.household_ref,
                               "left_semi")

occupancy = (bronze("bronze_pms_occupancy").filter(F.col("run_id") == RUN_ID)
             .select(
                 F.col("event_id"), F.col("unit_id"),
                 F.col("household_ref").alias("household_key"),
                 F.to_date("occupied_from").alias("occupied_from"),
                 F.to_date("occupied_to").alias("occupied_to"),
                 F.col("vacate_reason"),
                 F.lit(RUN_ID).alias("run_id"),
                 F.lit(GENERATED_AT).alias("generated_at_utc")))

work_orders = (bronze("bronze_pms_work_orders").filter(F.col("run_id") == RUN_ID)
               .select(
                   F.col("work_order_id"), F.col("unit_ref").alias("unit_id"),
                   F.to_date("vacated_date").alias("vacated_date"),
                   F.to_date("ready_to_rent_date").alias("ready_to_rent_date"),
                   F.col("turnaround_category"),
                   F.lit(RUN_ID).alias("run_id"),
                   F.lit(GENERATED_AT).alias("generated_at_utc")))

known_units = units.select("unit_id").distinct()
orphan_orders = work_orders.join(known_units, "unit_id", "left_anti")
orphan_order_count = orphan_orders.count()
if orphan_order_count:
    quarantine(orphan_orders, "silver_fact_unit_turnaround",
               "unit_ref does not resolve to a known unit")
check("silver_fact_unit_turnaround", "unit_id resolves to a known unit",
      work_orders.count() - orphan_order_count, orphan_order_count)
work_orders_valid = work_orders.join(known_units, "unit_id", "left_semi")

# Turnaround days is only meaningful once the unit is ready; an open work order is
# genuinely unknown, not zero.
turnaround = work_orders_valid.withColumn(
    "turnaround_days",
    F.when(F.col("ready_to_rent_date").isNotNull(),
           F.datediff("ready_to_rent_date", "vacated_date")).otherwise(F.lit(None)))
still_open = turnaround.filter(F.col("turnaround_days").isNull()).count()
check("silver_fact_unit_turnaround", "turnaround_days computed where unit is ready",
      turnaround.count() - still_open, still_open,
      "open work orders left null - an unfinished turnaround is unknown, not zero")

In [ ]:
def write_run_scoped(frame, table_name):
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (frame.write.format("delta").mode("overwrite")
            .partitionBy("run_id").saveAsTable(table_name))
    except Exception as error:
        print(f"  WARNING {table_name} schema changed - replacing all runs "
              f"({str(error).splitlines()[0][:100]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (frame.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    print(f"  {table_name:34} {frame.count():>9,} rows")


print("writing silver:")
write_run_scoped(buildings, "silver_dim_building")
write_run_scoped(units, "silver_dim_unit")
write_run_scoped(tenancies, "silver_dim_household")
write_run_scoped(charges, "silver_fact_rent_charge")
write_run_scoped(receipts_valid, "silver_fact_receipt")
write_run_scoped(occupancy, "silver_fact_occupancy")
write_run_scoped(turnaround, "silver_fact_unit_turnaround")

In [ ]:
QUALITY_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("rule", StringType(), True),
    StructField("rows_passed", IntegerType(), True),
    StructField("rows_failed", IntegerType(), True),
    StructField("pass_rate", DoubleType(), True),
    StructField("note", StringType(), True),
    StructField("checked_at_utc", StringType(), True),
])
QUARANTINE_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("record", StringType(), True),
    StructField("quarantined_at_utc", StringType(), True),
])


def frame_of(rows, schema):
    columns = [field.name for field in schema.fields]
    return spark.createDataFrame(
        [tuple(row.get(column) for column in columns) for row in rows], schema=schema)


write_run_scoped(frame_of(quality_rows, QUALITY_SCHEMA), "silver_data_quality")
write_run_scoped(frame_of(quarantine_rows or [], QUARANTINE_SCHEMA)
                 if quarantine_rows else
                 spark.createDataFrame([], QUARANTINE_SCHEMA), "silver_quarantine")

print("\ndata quality:")
for row in quality_rows:
    flag = "  " if row["rows_failed"] == 0 else "!!"
    print(f" {flag} {row['table_name']:30} {row['rule'][:46]:48} "
          f"pass {row['pass_rate']:.2%}  failed {row['rows_failed']:,}")

In [ ]:
display(spark.read.table("silver_data_quality").filter(F.col("run_id") == RUN_ID)
        .select("table_name", "rule", "rows_passed", "rows_failed", "pass_rate", "note"))